# 04a — Video Mode 2: Character Replacement (Wan2.2-Animate-14B, primary path)

Replaces the ComfyUI `04_video_wan_vace_animate.json` workflow. Pure diffusers: `WanAnimatePipeline`
in **replace mode** — takes a source video + your character still, and re-renders the source subject
as *your* character while preserving the original scene, pose, motion, and timing.

**Pipeline** (per the official Wan2.2 Animate flow):
1. **Preprocess** the source clip with the repo's `preprocess_data.py` → extracts `src_face.mp4`,
   `src_pose.mp4`, and (replace mode) `src_bg.mp4` + `src_mask.mp4`.
2. **Run** `WanAnimatePipeline(..., mode="replace")` with the character image + those 4 videos.

**Important (from the Wan-Animate UserGuider):**
- Replacement mode's built-in mask extractor is **single-person only**. Multi-subject source clips
  need a custom mask (see the VACE notebook 04b for the SAM2 route, or bring your own mask video).
- If your character's body proportions differ a lot from the source subject, expect edge artifacts
  (retargeting is intentionally OFF in replace mode to protect background interaction).

**VRAM:** ~28 GB (bf16) + preprocessing models. A100-40 with offloading, A100-80 resident.

**Inputs:** source video (single person), a clean front-facing character still (02a output works
well — full upper body, neutral background is best).

## 1. Config + mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── Config — change these ────────────────────────────────────────────────
CHARACTER_NAME = 'Yuna'
TRIGGER_TOKEN  = 'sks_vyuna'
RES_AREA_W, RES_AREA_H = 1280, 720   # resolution area for preprocess + generation
SEGMENT_FRAMES = 77                  # 4k+1; longer clips auto-segment (slow)
# ─────────────────────────────────────────────────────────────────────────

import os
DRIVE_BASE = '/content/drive/MyDrive/ai_character_studio'
VID_OUT    = f'{DRIVE_BASE}/outputs/videos/{CHARACTER_NAME}/mode2_animate'
os.makedirs(VID_OUT, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf_cache'   # LOCAL disk — a Drive-backed HF cache corrupts large gated models (FUSE atomic-rename); re-download each session
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

# Set these before cell 4 (upload below or point at Drive):
SOURCE_VIDEO  = None   # e.g. '/content/drive/MyDrive/ai_character_studio/inputs/source_clip.mp4'
CHARACTER_REF = None   # e.g. a 02a full-body still, or a training reference image

print(f'Video out: {VID_OUT}')
print('Set SOURCE_VIDEO + CHARACTER_REF (upload cells below, or paths to Drive files).')

## 2. Upload inputs (or use Drive paths)

In [ ]:
# Upload the source clip (single person!) and a character reference still.
from google.colab import files
import os
UP_DIR = f'{DRIVE_BASE}/inputs'
os.makedirs(UP_DIR, exist_ok=True)

print('Upload the SOURCE VIDEO (.mp4):')
vid = files.upload()
SOURCE_VIDEO = list(vid)[0]
print('Now upload the CHARACTER REFERENCE still (.png/.jpg):')
ref = files.upload()
CHARACTER_REF = list(ref)[0]
print('SOURCE_VIDEO  =', SOURCE_VIDEO)
print('CHARACTER_REF =', CHARACTER_REF)

## 3. Install deps + set up the Wan2.2 repo (isolated uv venv, like 01c)
The preprocessing script comes from the Wan2.2 GitHub repo and has its own dependency set
(onnxruntime for pose/sam2, etc.), so it runs in its own venv to keep Colab's torch intact.
The **diffusers pipeline itself** (cell 6) runs in Colab's main env — only preprocessing is isolated.

In [ ]:
import os, subprocess
WAN22 = '/content/Wan2.2'
VENV  = f'{WAN22}/.venv'
PY    = f'{VENV}/bin/python'

!pip install -q uv

if not os.path.exists(WAN22):
    !git clone --depth 1 https://github.com/Wan-Video/Wan2.2.git {WAN22}

# Isolated venv (own Python) for preprocessing. torch from cu124 (matches Colab), then the repo reqs.
!uv venv --python 3.11 {VENV}
!uv pip install --python {PY} "torch==2.7.1" "torchvision==0.22.1" --index-url https://download.pytorch.org/whl/cu124
!uv pip install --python {PY} -r {WAN22}/requirements.txt
# Preprocessing-specific extras (pose/sam2/onnx) if not already in requirements
!uv pip install --python {PY} onnxruntime-gpu av pillow

# Sanity: the preprocess script must exist
pre = f'{WAN22}/wan/modules/animate/preprocess/preprocess_data.py'
print('preprocess_data.py present:', os.path.exists(pre))
if not os.path.exists(pre):
    print('  (the repo layout may have changed — locate it with: )')
    !find {WAN22} -name 'preprocess_data.py'

# Also make sure the main (Colab) env has the diffusers stack for the pipeline in cell 6
!uv pip install --system -q --reinstall-package diffusers \
    "diffusers>=0.35.0" transformers accelerate safetensors huggingface_hub ftfy
print('✅ envs ready (isolated venv for preprocess, system diffusers for the pipeline).')

# torchao: Colab ships 0.10.0 but the peft uv pulls requires >=0.16.0 and its
# is_torchao_available() check raises instead of returning False. We don't use
# torchao (quantization lib) for bf16 FLUX/Wan/LTX + LoRA, so remove it to keep
# peft's LoRA injection path clean.
!uv pip uninstall --system -y torchao 2>/dev/null || true
# ── CUDA-torch guard ────────────────────────────────────────────────────────
# uv can silently re-resolve deps and swap Colab's CUDA torch for the PyPI CPU
# wheel ("Torch not compiled with CUDA enabled" at load time). Detect + repair
# here, BEFORE we commit to a multi-GB model download.
import torch as _t
print('torch', _t.__version__, '| cuda', _t.cuda.is_available())
if not _t.cuda.is_available():
    import subprocess
    ver = _t.__version__.split('+')[0]
    print(f'CPU-only torch {ver} detected (uv swapped it). Reinstalling the CUDA build from cu124...')
    subprocess.run(f'uv pip install --system "torch=={ver}" torchvision '
                   '--index-url https://download.pytorch.org/whl/cu124',
                   shell=True, check=True)
    raise RuntimeError(
        'CUDA torch reinstalled on disk. Now: Runtime > Restart runtime, then re-run '
        'this install cell + the model-load cell. (The running kernel still holds the '
        'old CPU torch in memory, so the restart is required — do not skip it.)')


## 4. Preprocessing — download the Animate checkpoint + run `preprocess_data.py`
First run downloads `Wan-AI/Wan2.2-Animate-14B` (~30 GB) into `WAN22/Wan2.2-Animate-14B`. The
preprocessor also needs its small detector/pose/sam2 weights — it downloads them into
`process_checkpoint/` on first run (see UserGuider for the expected `det/`, `pose2d/`, `sam2/`
layout). Log goes to a file.

Outputs land in `PREPROC_OUT`: `src_face.mp4`, `src_pose.mp4`, `src_bg.mp4`, `src_mask.mp4`.

In [ ]:
import subprocess, os, time

MODEL_LOCAL = f'{WAN22}/Wan2.2-Animate-14B'
PREPROC_OUT = f'{DRIVE_BASE}/mode2_preproc/{CHARACTER_NAME}_{time.strftime("%Y%m%d_%H%M%S")}'
os.makedirs(PREPROC_OUT, exist_ok=True)

# 1) download the model (huggingface-cli, into the repo-local dir the script expects)
if not os.path.isdir(MODEL_LOCAL):
    print('Downloading Wan2.2-Animate-14B (~30GB) → Drive HF cache →', MODEL_LOCAL)
    env = {**os.environ, 'HF_HOME': os.environ['HF_HOME'], 'HF_HUB_ENABLE_HF_TRANSFER': '1'}
    subprocess.run([
        'huggingface-cli', 'download', 'Wan-AI/Wan2.2-Animate-14B',
        '--local-dir', MODEL_LOCAL
    ], check=True, env=env)
else:
    print('Animate model already present.')

# 2) run the OFFICIAL replacement-mode preprocessing
LOG = f'{PREPROC_OUT}/preprocess.log'
cmd = [
    PY, f'{WAN22}/wan/modules/animate/preprocess/preprocess_data.py',
    '--ckpt_path', f'{MODEL_LOCAL}/process_checkpoint',
    '--video_path', SOURCE_VIDEO,
    '--refer_path', CHARACTER_REF,
    '--save_path', PREPROC_OUT,
    f'--resolution_area', str(RES_AREA_W), str(RES_AREA_H),
    '--iterations', '3',
    '--k', '7',
    '--w_len', '1',
    '--h_len', '1',
    '--replace_flag',
]
print('Running replacement preprocessing (log →', LOG, ')')
with open(LOG, 'w') as logf:
    r = subprocess.run(cmd, cwd=WAN22, stdout=logf, stderr=subprocess.STDOUT, text=True)
print('preprocess exit code:', r.returncode)
print('--- last 25 log lines ---')
print(subprocess.run(['tail', '-n', '25', LOG], capture_output=True, text=True).stdout)

# 3) confirm the 4 expected videos
need = ['src_face.mp4', 'src_pose.mp4', 'src_bg.mp4', 'src_mask.mp4']
for f in need:
    p = f'{PREPROC_OUT}/{f}'
    print(f'  {f}:', 'OK' if os.path.exists(p) else 'MISSING')
if not all(os.path.exists(f'{PREPROC_OUT}/{f}') for f in need):
    print('⚠️  Some expected outputs missing. Check the log — multi-person sources fail the mask step.')
    print('    Inspect:', PREPROC_OUT)

## 5. (Review) eyeball the extracted mask before the expensive run
The mask defines *what gets replaced*. A bad mask = bad result. Quick montage of a few frames.

In [ ]:
# Optional sanity check: show a few frames of src_mask.mp4 (white = region to regenerate).
import subprocess
from PIL import Image
from IPython.display import display, HTML
import base64

tmp = f'{PREPROC_OUT}/_maskcheck'
os.makedirs(tmp, exist_ok=True)
subprocess.run(['ffmpeg', '-y', '-i', f'{PREPROC_OUT}/src_mask.mp4', '-vf', 'select=eq(n\,0)+eq(n\,20)+eq(n\,40)',
                '-vsync', 'vfr', f'{tmp}/m_%02d.png'], capture_output=True)
frames = sorted(os.listdir(tmp))
def b64p(p):
    im = Image.open(p).convert('RGB'); im.thumbnail((320, 320))
    from io import BytesIO
    b = BytesIO(); im.save(b, 'JPEG', quality=80)
    return f'<img src="data:image/jpeg;base64,{base64.b64encode(b.getvalue()).decode()}" style="border:1px solid #555">'
display(HTML('<h4>Mask frames (white = regenerate)</h4><div style="display:flex;gap:6px">' +
             ''.join(b64p(f'{tmp}/{f}') for f in frames) + '</div>'))
print('If the mask is wrong/empty, rerun cell 4 with different --iterations / --k (bigger) or '
      '--w_len/--h_len. Or supply your own mask video (see §8).')

## 6. Load `WanAnimatePipeline` and run replacement
VAE in fp32 (decode quality), transformer bf16. Group offloading below 60 GB.

In [ ]:
import torch, logging, numpy as np, time
from diffusers import AutoencoderKLWan, WanAnimatePipeline
from diffusers.utils import export_to_video, load_image, load_video
from pathlib import Path

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU ~{vram_gb:.0f} GB')
LOG = '/content/animate_load.log'
logging.basicConfig(filename=LOG, level=logging.INFO)

model_id = "Wan-AI/Wan2.2-Animate-14B-Diffusers"
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", dtype=torch.float32)
pipe = WanAnimatePipeline.from_pretrained(model_id, vae=vae, dtype=torch.bfloat16)

if vram_gb >= 60:
    pipe.to('cuda'); print('Strategy: resident')
else:
    from diffusers.hooks import apply_group_offloading
    onload, offload = torch.device('cuda'), torch.device('cpu')
    apply_group_offloading(pipe.text_encoder, onload_device=onload, offload_device=offload,
                           offload_type='block_level', num_blocks_per_group=4)
    pipe.transformer.enable_group_offload(onload_device=onload, offload_device=offload,
                                          offload_type='leaf_level', use_stream=True)
    print('Strategy: group offloading')

print('✅ WanAnimatePipeline ready. Log →', LOG)

In [ ]:
# ── Run replacement ────────────────────────────────────────────────────────
def aspect_ratio_resize(image, pipe, max_area=1280*720):
    ar = image.height / image.width
    mod = pipe.vae_scale_factor_spatial * pipe.transformer.config.patch_size[1]
    h = round(np.sqrt(max_area * ar)) // mod * mod
    w = round(np.sqrt(max_area / ar)) // mod * mod
    return image.resize((w, h)), h, w

image = load_image(CHARACTER_REF).convert('RGB')
image, h, w = aspect_ratio_resize(image, pipe)

pose_video   = load_video(f'{PREPROC_OUT}/src_pose.mp4')
face_video   = load_video(f'{PREPROC_OUT}/src_face.mp4')
bg_video     = load_video(f'{PREPROC_OUT}/src_bg.mp4')
mask_video   = load_video(f'{PREPROC_OUT}/src_mask.mp4')

prompt   = "A person seamlessly integrated into the scene with consistent lighting and environment"
negative = "blurry, low quality, inconsistent lighting, floating, disconnected from scene"

g = torch.Generator(device='cuda').manual_seed(0)
out = pipe(
    image=image,
    pose_video=pose_video,
    face_video=face_video,
    background_video=bg_video,
    mask_video=mask_video,
    prompt=prompt,
    negative_prompt=negative,
    height=h, width=w,
    segment_frame_length=SEGMENT_FRAMES,
    prev_segment_conditioning_frames=5,   # 5 = better temporal consistency (more VRAM)
    guidance_scale=1.0,                    # CFG off by default for Animate
    generator=g,
).frames[0]

ts = time.strftime('%Y%m%d_%H%M%S')
out_path = Path(VID_OUT) / f'{ts}_replace.mp4'
export_to_video(out, str(out_path), fps=30)
print(f'✅ replaced video → {out_path}')

from IPython.display import Video, display
display(Video(str(out_path), width=720))

## 7. Log to metadata

In [ ]:
import json, os, glob, time
meta_path = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}/metadata.json'
os.makedirs(os.path.dirname(meta_path), exist_ok=True)
meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {'name': CHARACTER_NAME}
clips = sorted(glob.glob(f'{VID_OUT}/*.mp4'))
meta.setdefault('video_log', []).append({
    'ts': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'mode': 'mode2_animate',
    'model': 'Wan2.2-Animate-14B (replace)',
    'clips': clips[-5:],
})
json.dump(meta, open(meta_path, 'w'), indent=2)
print(f'metadata.json updated — {len(clips)} clips in {VID_OUT}')

## 8. Commented alternates (try later)
The other Mode 2 variants + levers.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# A) ANIMATE mode (no source background) — your character reenacts the source
#    person's motion/expression on a generated (or black) background. Drop the
#    background_video/mask_video args, set mode="animate".
# ─────────────────────────────────────────────────────────────────────────
# out = pipe(image=image, pose_video=pose_video, face_video=face_video,
#            prompt=prompt, negative_prompt=negative, height=h, width=w,
#            segment_frame_length=SEGMENT_FRAMES, guidance_scale=1.0,
#            mode="animate").frames[0]

# ─────────────────────────────────────────────────────────────────────────
# B) Relighting LoRA — Wan-Animate's auxiliary module that keeps the character's
#    appearance while matching the scene's lighting/color tone. Load onto the
#    transformer for more convincing integration.
# ─────────────────────────────────────────────────────────────────────────
# pipe.load_lora_weights("Wan-AI/Wan2.2-Animate-14B-Relighting-LoRA", adapter_name="relight")
# pipe.set_adapters(["relight"])

# ─────────────────────────────────────────────────────────────────────────
# C) CFG ON (guidance_scale 5.0) — Animate defaults CFG off; turning it on gives
#    stronger text/face-prompt control at a quality/consistency trade.
# ─────────────────────────────────────────────────────────────────────────
# out = pipe(..., guidance_scale=5.0).frames[0]

# ─────────────────────────────────────────────────────────────────────────
# D) LightX2V 4-step distillation — big speed win for Animate iteration.
# ─────────────────────────────────────────────────────────────────────────
# pipe.load_lora_weights("lightx2v/...-4steps", adapter_name="fast"); pipe.set_adapters(["fast"])

# ─────────────────────────────────────────────────────────────────────────
# E) Bring your OWN mask video (multi-person sources, or a hand-made mask).
#    Replace the src_mask.mp4 path in cell 6's load_video call. White=regenerate.
# ─────────────────────────────────────────────────────────────────────────
# mask_video = load_video('path/to/my_mask.mp4')

# ─────────────────────────────────────────────────────────────────────────
# F) Lower-VRAM alternate: see 04b_test_video_mode2_vace.ipynb (Wan VACE 1.3B,
#    SAM2 masking, swap-anything) — ~8 GB at 480p.
# ─────────────────────────────────────────────────────────────────────────
print('Section 8: alternates commented out.')